This is a minimal interactive example for trying demonstrating different models trained on a variety of datasets on a few typical examples.

In [13]:
import seisbench.models as sbm
import numpy as np
import matplotlib.pyplot as plt
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from ipywidgets import interact, widgets
from IPython.display import display

import seisbench
seisbench.__version__ = "0.13.0"  # Overwrite SeisBench version if you want to use very recent models

In [76]:
# Load examples

# Chile
client = Client("GFZ")

t = UTCDateTime("2007/01/02 05:48:50")
stream_chile = client.get_waveforms(
    network="CX",
    station="PB01",
    location="*",
    channel="HH?",
    starttime=t - 30,
    endtime=t + 150,
)

# OBS
client = Client("IRIS")

t0 = UTCDateTime(1423177269.5831)
stream_obs = client.get_waveforms(network="YH", station="LOBS8", location="*", channel="H??", starttime=t0, endtime=t0 + 200)

# Ridgecrest
client = Client("SCEDC")

starttime = UTCDateTime("2019-07-06T08:28:00")
endtime = starttime + 5 * 60

stream_rigdecrest = client.get_waveforms(
    network="CI",
    station="PASC",
    location="10",
    channel="HH?",
    starttime=starttime,
    endtime=endtime,
)

examples = [
    ("Chile (IPOC)", stream_chile),
    ("Ridgecrest", stream_rigdecrest),
    ("OBS (HOBITSS)", stream_obs)
]

In [86]:
# Preload all metadata
available = {
    "PhaseNet": sbm.PhaseNet,
    "EQTransformer": sbm.EQTransformer,
    "EQTP": sbm.EQTP,
    "EQCCTP": sbm.EQCCTP,
    "EQCCTS": sbm.EQCCTS,
    "OBSTransformer": sbm.OBSTransformer,
    "DKPN": sbm.DKPN,
    "SeisT": sbm.SeisT,
    # "SkyNet": sbm.Skynet,
}

for model in available.values():
    model.list_pretrained(remote=True)

In [90]:
def plot_example(model, stream, weight, overlap):
    model = model.from_pretrained(weight)
        
    print(model.weights_docstring)
    annotations = model.annotate(stream, overlap=overlap)
    picks = model.classify(stream, overlap=overlap).picks
    fig = plt.figure(figsize=(12, 8))
    axs = fig.subplots(2, 1, sharex=True, gridspec_kw={"hspace": 0})

    for trace in stream:
        trace.data = trace.data - np.mean(trace.data)
    
    t0 = min(trace.stats.starttime for trace in stream)
    t1 = max(trace.stats.endtime for trace in stream)
    amplitude = max(np.max(np.abs(trace.data)) for trace in stream)
    for i, trace in enumerate(stream):
        offset = trace.stats.starttime - t0
        axs[0].plot(trace.times() + offset, trace.data + 1.5 * i * amplitude, lw=0.5)
        axs[0].text(0,  (1.5 * i + 0.2) * amplitude, " " + trace.id, c=f"C{i}", weight="bold")

    axs[0].set_ylim(- 1.1 * amplitude, (1.5 * len(stream) - 0.4) * amplitude)
    axs[0].set_xlim(0, t1 - t0)

    for ann in annotations:
        if any(ann.id.endswith(x) for x in "NUD"):
            continue
        offset = ann.stats.starttime - t0
        axs[1].plot(ann.times() + offset, ann.data, label=ann.id.split("_")[-1])
    axs[1].legend(loc="upper right")
    
    axs[1].set_ylim(0, 1)

    for pick in picks:
        axs[0].axvline(pick.peak_time - t0, ls=":", lw=1, c="k")
        axs[0].text(pick.peak_time - t0, 0.99, " " + pick.phase, transform=axs[0].get_xaxis_transform(), va="top")

    axs[1].set_xlabel("Time [s]")
    axs[0].set_yticks([])
    axs[0].set_ylabel("Velocity [raw]")
    axs[1].set_ylabel("Pick confidence")

In [91]:
model = widgets.Dropdown(
    options=list(available.items()),
    description="Model:",
)

dataset = widgets.Dropdown(
    options=model.value.list_pretrained(remote=False),
    description="Training dataset:",
)

data = widgets.Dropdown(
    options=examples,
    description="Example: ",
)

overlap = widgets.FloatSlider(
    value=0.5,
    min=0.20,
    max=0.95,
    step=0.05,
    description="Overlap:",
    readout_format=".2f",
    continuous_update=False,
)

def update_datasets(change):
    options = change["new"].list_pretrained(remote=False)
    dataset.options = options
    dataset.value = options[0]

model.observe(update_datasets, names="value")

output = widgets.interactive_output(
    plot_example,
    {"model": model, "weight": dataset, "stream": data, "overlap": overlap},
)

display(
    widgets.HBox([data, model, dataset, overlap]),
    output,
)

Output(outputs=({'name': 'stderr', 'text': 'Exception in callback Task.__step()\nhandle: <Handle Task.__step()…